# 03 — Fusión de triage, respuesta automatizada y demo con Voilà
### Tema 6 — Sistemas expertos en SOC | IA Aplicada a la Ciberseguridad

Este notebook es el **sistema experto central**: carga los dos modelos entrenados en los notebooks 01 y 02,
define una escala de severidad unificada, simula alertas entrantes de ambos tipos, y dispara la respuesta
automatizada correspondiente vía un webhook de Shuffle (SOAR).

Ejecuta este notebook con:

```bash
voila 03_triage_fusion_soar_demo.ipynb
```

para obtener un dashboard limpio (sin código visible) para la sustentación en vivo.
Documentación de Voilà: https://voila.readthedocs.io/
Documentación de Shuffle: https://shuffler.io/docs


## 1. Instalación de dependencias

In [ ]:
# !pip install ipywidgets voila requests joblib scikit-learn xgboost --quiet


## 2. Imports y carga de modelos

Copia los archivos `.joblib` generados por los notebooks 01 y 02 en el mismo directorio que este notebook antes de ejecutar.

In [ ]:
import numpy as np
import joblib
import requests
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

# Modelo de phishing (notebook 01)
phishing_model = joblib.load("phishing_model.joblib")
phishing_scaler = joblib.load("phishing_scaler.joblib")
phishing_label_encoder = joblib.load("phishing_label_encoder.joblib")
phishing_features = joblib.load("phishing_selected_features.joblib")

# Modelo de privilege escalation / unauthorized access (notebook 02)
privesc_model = joblib.load("privesc_model.joblib")
privesc_scaler = joblib.load("privesc_scaler.joblib")
privesc_features = joblib.load("privesc_feature_cols.joblib")

print("Modelos cargados correctamente.")


## 3. Mapeo MITRE ATT&CK y niveles de severidad unificados

Aquí es donde se materializa el "sistema experto": una tabla de reglas que traduce la salida probabilística
de cada modelo en una severidad accionable, ligada a la técnica MITRE ATT&CK correspondiente.

In [ ]:
MITRE_MAP = {
    "phishing": {
        "tecnica": "T1566 / T1566.001 / T1204.002",
        "nombre": "Phishing - Spearphishing Attachment - User Execution",
        "url": "https://attack.mitre.org/techniques/T1566/",
    },
    "unauthorized_access": {
        "tecnica": "T1078 / T1548",
        "nombre": "Valid Accounts - Abuse Elevation Control Mechanism",
        "url": "https://attack.mitre.org/techniques/T1078/",
    },
}

def severidad_desde_probabilidad(proba, tipo):
    # Convierte la probabilidad del modelo en un nivel de severidad accionable
    if tipo == "phishing":
        if proba < 0.4:
            return "Bajo"
        elif proba < 0.7:
            return "Medio"
        elif proba < 0.9:
            return "Alto"
        else:
            return "Critico"
    else:  # unauthorized_access
        # Umbrales mas bajos porque el dataset es extremadamente desbalanceado:
        # cualquier probabilidad no trivial de acceso no autorizado ya es relevante
        if proba < 0.2:
            return "Bajo"
        elif proba < 0.5:
            return "Medio"
        elif proba < 0.8:
            return "Alto"
        else:
            return "Critico"

acciones_por_severidad = {
    "Bajo": "Registrar en log, sin accion automatica",
    "Medio": "Crear ticket en cola SOC para revision manual",
    "Alto": "Forzar cierre de sesion / poner cuenta en cuarentena y notificar analista de guardia",
    "Critico": "Bloquear cuenta, aislar host, revocar tokens y escalar a Tier 2 de inmediato",
}


## 4. Integración con Shuffle (SOAR)

Reemplaza `SHUFFLE_WEBHOOK_URL` con la URL real de tu workflow (Shuffle -> Workflow -> Webhook trigger).

In [ ]:
SHUFFLE_WEBHOOK_URL = "https://your-shuffle-instance/api/v1/hooks/webhook_id"  # reemplazar

def disparar_respuesta(tipo, severidad, detalle="", enviar_real=False):
    accion = acciones_por_severidad.get(severidad, "Sin accion definida")
    payload = {
        "tipo_incidente": tipo,
        "tecnica_mitre": MITRE_MAP[tipo]["tecnica"],
        "severidad": severidad,
        "accion_recomendada": accion,
        "detalle": detalle,
    }

    if enviar_real:
        try:
            resp = requests.post(SHUFFLE_WEBHOOK_URL, json=payload, timeout=5)
            print(f"Webhook enviado a Shuffle -> status {resp.status_code}")
        except Exception as e:
            print(f"No se pudo contactar Shuffle: {e}")
    else:
        print(f"[SIMULADO] {tipo} | Severidad={severidad} | MITRE={MITRE_MAP[tipo]['tecnica']}")
        print(f"           Accion: {accion}")

    return payload


## 5. Funciones de clasificación por tipo de alerta

In [ ]:
def clasificar_url_phishing(fila_features):
    # fila_features: array 1D en el mismo orden que phishing_features
    fila_scaled = phishing_scaler.transform(fila_features.reshape(1, -1))
    proba = phishing_model.predict_proba(fila_scaled)[0][1]
    severidad = severidad_desde_probabilidad(proba, "phishing")
    return severidad, proba

def clasificar_evento_acceso(fila_features):
    # fila_features: array 1D en el mismo orden que privesc_features
    fila_scaled = privesc_scaler.transform(fila_features.reshape(1, -1))
    proba = privesc_model.predict_proba(fila_scaled)[0][1]
    severidad = severidad_desde_probabilidad(proba, "unauthorized_access")
    return severidad, proba


## 6. Simulación de alertas entrantes (paso 5 del guion de sustentación)

Simula un flujo mixto de alertas —algunas de phishing, algunas de acceso no autorizado— procesadas en
tiempo real por el sistema experto. Reemplaza los arrays de ejemplo por filas reales de tus sets de
prueba (`X_test_scaled` de los notebooks 01 y 02) antes de la demo final.

In [ ]:
# Ejemplo ilustrativo: reemplazar por filas reales de X_test de cada notebook
ejemplo_url_features = np.random.rand(len(phishing_features))
ejemplo_acceso_features = np.random.rand(len(privesc_features))

print("=== Simulacion de alertas entrantes ===\n")

severidad, proba = clasificar_url_phishing(ejemplo_url_features)
print(f"[Alerta 1] URL sospechosa | Severidad: {severidad} | Confianza: {proba:.2%}")
disparar_respuesta("phishing", severidad, detalle="URL reportada por gateway de correo", enviar_real=False)

time.sleep(0.5)
print()

severidad, proba = clasificar_evento_acceso(ejemplo_acceso_features)
print(f"[Alerta 2] Patron de autenticacion anomalo | Severidad: {severidad} | Confianza: {proba:.2%}")
disparar_respuesta("unauthorized_access", severidad, detalle="Multiples destinos distintos en una hora", enviar_real=False)


## 7. Dashboard interactivo para Voilà

Botones independientes para cada tipo de alerta — así el jurado ve ambas ramas de la arquitectura
funcionando durante la sustentación en vivo.

In [ ]:
boton_phishing = widgets.Button(description="Simular alerta de phishing", button_style="warning")
boton_acceso = widgets.Button(description="Simular evento de acceso", button_style="danger")
salida = widgets.Output()

def al_simular_phishing(b):
    with salida:
        clear_output()
        fila = np.random.rand(len(phishing_features))  # reemplazar por muestra real
        severidad, proba = clasificar_url_phishing(fila)
        print(f"Tipo: Phishing (MITRE {MITRE_MAP['phishing']['tecnica']})")
        print(f"Severidad: {severidad}  (confianza {proba:.2%})")
        disparar_respuesta("phishing", severidad, enviar_real=False)

def al_simular_acceso(b):
    with salida:
        clear_output()
        fila = np.random.rand(len(privesc_features))  # reemplazar por muestra real
        severidad, proba = clasificar_evento_acceso(fila)
        print(f"Tipo: Unauthorized Access (MITRE {MITRE_MAP['unauthorized_access']['tecnica']})")
        print(f"Severidad: {severidad}  (confianza {proba:.2%})")
        disparar_respuesta("unauthorized_access", severidad, enviar_real=False)

boton_phishing.on_click(al_simular_phishing)
boton_acceso.on_click(al_simular_acceso)

display(widgets.HBox([boton_phishing, boton_acceso]), salida)


## Checklist antes de la sustentación

- [ ] Reemplazar los arrays aleatorios de ejemplo por filas reales de `X_test` de cada notebook.
- [ ] Configurar `SHUFFLE_WEBHOOK_URL` con el workflow real desplegado en Shuffle.
- [ ] Ejecutar `voila 03_triage_fusion_soar_demo.ipynb` una vez antes del examen para verificar que corre
      sin errores fuera de JupyterLab.
- [ ] Tener capturas de pantalla de respaldo por si falla la conexión en vivo el día del examen.
